In [6]:
from docling.document_converter import DocumentConverter


import fitz  # PyMuPDF

def redact_hyperlinked_text(input_path, output_path):
    doc = fitz.open(input_path)
    
    for page in doc:
        # get_links() returns a list of dictionaries representing all clickable areas
        links = page.get_links()
        
        for link in links:
            # link["from"] contains the exact bounding box (fitz.Rect) of the hyperlink
            link_rect = link["from"]
            
            # Add a redaction annotation over this specific box, filled with white
            page.add_redact_annot(link_rect, fill=(1, 1, 1))
        
        # Apply the redactions, physically wiping the text under the boxes
        page.apply_redactions()
        
    doc.save(output_path)
    doc.close()
    print("All hyperlinked elements redacted. Clean PDF saved.")


redact_hyperlinked_text("./data/document.pdf", "./data/clean_document.pdf")
# redact_superscripts_dynamically("./data/document.pdf", "./data/clean_document.pdf")

source = "./data/clean_document.pdf"  # document per local path or URL
converter = DocumentConverter()
result = converter.convert(source)

# 1. Get the full markdown string
str_result = result.document.export_to_markdown()

# 2. Split the document into individual structural blocks (separated by double newlines)
blocks = str_result.split('\n\n')
extracted_data = []

for i, block in enumerate(blocks):
    # A standard Markdown table contains a header separator row like "|---"
    if '|---' in block or '| ---' in block:
        extracted_data.append("### --- NEW TABLE EXTRACTION --- ###")
        
        # Grab the paragraph immediately before the table (often the caption or introduction)
        if i > 0:
            prev_block = blocks[i-1].strip()
            if prev_block:
                extracted_data.append(f"**[Preceding Paragraph]**\n{prev_block}")
        
        # Add the table itself
        extracted_data.append(f"**[Table Data]**\n{block.strip()}")
        
        # Grab the paragraph immediately after the table (often table footnotes or continuation)
        if i + 1 < len(blocks):
            next_block = blocks[i+1].strip()
            if next_block:
                extracted_data.append(f"**[Succeeding Paragraph]**\n{next_block}")
        
        extracted_data.append("\n") # Add spacing between different table extractions

# 3. Join the extracted blocks back into a single string
final_output = '\n\n'.join(extracted_data)

print(final_output)

with open('out.txt', 'w', encoding="utf-8") as output:
    output.write(final_output)

All hyperlinked elements redacted. Clean PDF saved.


[INFO] 2026-06-23 14:06:31,665 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 14:06:31,693 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-06-23 14:06:31,694 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-06-23 14:06:31,982 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 14:06:31,985 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-06-23 14:06:31,986 [RapidOCR] main.py:53: Using C:\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-06-23 14:06:32,086 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-23 14:06:32,100 [RapidOCR] download_file.py:60: File exists and is valid: C:\Python312\Lib\site-packages\rapidocr\mode

### --- NEW TABLE EXTRACTION --- ###

**[Preceding Paragraph]**
Table 1. Mark -Houwink Parameters K and a , Flory Parameters b , and Constants c of the Molar Mass Dependences Determined via DOSY for PS, PEO, and PMMA as Well as Dynamic Viscosities

**[Table Data]**
| polymer   | solvent      |   K [mL · g - 1 ] |       a |        b |   c · 10 - 8 [m 2 · s - 1 |   η [mPa · s] |
|-----------|--------------|-------------------|---------|----------|---------------------------|---------------|
| PS        | THF-d8       |            0.0141 |     0.7 | 0.519361 |                   2.20882 |          0.48 |
|           | CDCl 3       |           0.00732 | 0.74596 | 0.534677 |                   2.34303 |         0.563 |
|           | C 6 D 6      |           0.01077 | 0.72682 |   0.5283 |                   1.91203 |        0.6067 |
|           | toluene- d 8 |            0.0098 | 0.73512 | 0.531065 |                   2.13758 |          0.56 |
|           | CD 2 Cl 2    |           0.01155 | 0

In [ ]:
import fitz  # PyMuPDF

def redact_superscripts_dynamically(input_path, output_path):
    doc = fitz.open(input_path)
    
    for page in doc:
        page_dict = page.get_text("dict")
        
        for block in page_dict.get("blocks", []):
            if block.get("type") != 0:  # Ignore images/drawings
                continue
                
            for line in block.get("lines", []):
                spans = line.get("spans", [])
                if not spans:
                    continue
                
                # 1. Dynamically find the main font size and bottom baseline for THIS specific line
                main_font_size = max(span["size"] for span in spans)
                # bbox[3] is the bottom Y-coordinate of the bounding box
                line_bottom_y = max(span["bbox"][3] for span in spans) 
                
                for span in spans:
                    text = span["text"].strip()
                    font_size = span["size"]
                    span_bottom_y = span["bbox"][3]
                    
                    # 2. Check if the text is smaller than the line's main text
                    is_small = font_size < (main_font_size * 0.85)
                    
                    # 3. CRITICAL CHECK: Is it physically elevated above the baseline?
                    # If the bottom of the span is higher (smaller Y) than the line's bottom by at least 15% of the font size
                    is_elevated = span_bottom_y < (line_bottom_y - (main_font_size * 0.15))
                    
                    # 4. Is it formatted like a reference? (Digits or single letters like 'a', 'b')
                    is_ref_format = text.isdigit() or (len(text) == 1 and text.isalpha())
                    
                    if is_small and is_elevated and is_ref_format and text:
                        # This is a superscript! (Subscripts will fail the 'is_elevated' check)
                        bbox = fitz.Rect(span["bbox"])
                        page.add_redact_annot(bbox, fill=(1, 1, 1)) # White out the superscript
        
        page.apply_redactions()
        
    doc.save(output_path)
    doc.close()
    print("Elevated superscripts redacted. Subscripts preserved. Clean PDF saved.")

In [ ]:
from chemdataextractor import Document
from chemdataextractor.doc import Table

f = open("./data/document.html", 'rb')
raw_doc = Document.from_file(f)

# 1. Isolate only the Table elements
table_elements = [el for el in raw_doc.elements if isinstance(el, Table)]

# 2. Rebuild a Document composed entirely of tables
table_doc = Document(*table_elements)
print(table_doc.elements[0].serialize())  # print the first table in the document

# 3. Extract chemical relationships found exclusively in the tables
for record in table_doc.records:
    print(record.serialize())

In [19]:
import pandas as pd

# pandas will automatically find all <table> tags in the HTML 
# and return a list of DataFrames
tables = pd.read_html("./data/document.html")

print(f"Found {len(tables)} tables in the document.")

# Look at the first table
df = tables[0]
print(df.head())

# Convert it to a clean string format to paste into your LLM prompt
llm_input_string = df.to_markdown()
print(llm_input_string)

Found 1 tables in the document.
  polymer     solvent  Ka [mL·g–1]        aa        bb  c·10–8b [m2·s–1]  \
0      PS      THF-d8  0.0141 (27)  0.7 (27)  0.519361           2.20882   
1      PS       CDCl3      0.00732   0.74596  0.534677           2.34303   
2      PS        C6D6      0.01077   0.72682  0.528300           1.91203   
3      PS  toluene-d8       0.0098   0.73512  0.531065           2.13758   
4      PS      CD2Cl2      0.01155   0.70628  0.521453           2.74368   

     η [mPa·s]  
0    0.48 (21)  
1   0.563 (21)  
2  0.6067 (21)  
3    0.56 (21)  
4       0.413c  
|    | polymer   | solvent    | Ka [mL·g–1]   | aa       |       bb |   c·10–8b [m2·s–1] | η [mPa·s]   |
|---:|:----------|:-----------|:--------------|:---------|---------:|-------------------:|:------------|
|  0 | PS        | THF-d8     | 0.0141 (27)   | 0.7 (27) | 0.519361 |           2.20882  | 0.48 (21)   |
|  1 | PS        | CDCl3      | 0.00732       | 0.74596  | 0.534677 |           2.34303  | 0.5